# Part 1: Business Analysis - Is This Project Feasible?

## 🎯 Learning Objectives
- Understand the business problem BEFORE building solutions
- Explore raw data to assess project feasibility
- Identify data quality issues early
- Make data-driven recommendations

## 📊 Business Context

**Supermarket Challenge:**
- Fresh products have short shelf life
- Negative reviews indicate quality issues
- Food waste costs money and harms environment

**Proposed Solution:**
Use sentiment analysis on customer reviews to automatically recommend discounts for products with negative sentiment, helping to:
1. Reduce food waste
2. Improve customer satisfaction
3. Optimize pricing strategy

**Question:** Is this feasible with our data?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries loaded successfully!")

## Step 1: Load and Explore Raw Data

We have three data sources:
1. **Reviews** - Customer feedback (text + ratings)
2. **Inventory** - Stock levels over time
3. **Sales** - Sales volume over time

In [ ]:
# Load raw data
reviews_df = pd.read_csv('../data/raw/reviews.csv')
inventory_df = pd.read_csv('../data/raw/inventory.csv')
sales_df = pd.read_csv('../data/raw/sales.csv')

print("📦 Data Loaded:")
print(f"  Reviews: {len(reviews_df):,} rows")
print(f"  Inventory: {len(inventory_df):,} rows")
print(f"  Sales: {len(sales_df):,} rows")

In [ ]:
# First look at reviews
print("\n📝 REVIEWS DATA")
print("=" * 70)
reviews_df.head(10)

In [ ]:
# Data info
print("\n📊 Reviews Data Info:")
reviews_df.info()

## Step 2: Key Business Questions

Let's answer critical questions to determine feasibility:

In [ ]:
# Q1: Do we have enough review data per product?
print("\n❓ Q1: Review Volume by Product")
print("=" * 70)
review_counts = reviews_df['product_name'].value_counts()
print(review_counts)
print(f"\n💡 Analysis: Minimum {review_counts.min()} reviews per product")
print(f"   Average: {review_counts.mean():.0f} reviews per product")
if review_counts.min() > 50:
    print("   ✅ GOOD: Enough data for statistical significance")
else:
    print("   ⚠️  WARNING: May need more data for reliable predictions")

In [ ]:
# Q2: What's the sentiment distribution?
print("\n❓ Q2: Rating Distribution")
print("=" * 70)
print(reviews_df['rating'].value_counts().sort_index())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
reviews_df['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Overall Rating Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Rating (1-5 stars)')
axes[0].set_ylabel('Count')

# By product
reviews_df.groupby('product_name')['rating'].mean().sort_values().plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Average Rating by Product', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Average Rating')
axes[1].axvline(3, color='red', linestyle='--', alpha=0.5, label='Neutral (3.0)')
axes[1].legend()

plt.tight_layout()
plt.show()

# Create sentiment categories
def rating_to_sentiment(rating):
    if rating <= 2:
        return 'negative'
    elif rating >= 4:
        return 'positive'
    else:
        return 'neutral'

reviews_df['sentiment'] = reviews_df['rating'].apply(rating_to_sentiment)
sentiment_dist = reviews_df['sentiment'].value_counts(normalize=True) * 100

print(f"\n💡 Sentiment Distribution:")
for sent, pct in sentiment_dist.items():
    print(f"   {sent.upper()}: {pct:.1f}%")

if sentiment_dist.get('negative', 0) > 10:
    print(f"\n   ✅ GOOD: {sentiment_dist.get('negative', 0):.1f}% negative reviews - enough signal to optimize discounts")
else:
    print(f"\n   ⚠️  WARNING: Only {sentiment_dist.get('negative', 0):.1f}% negative - limited optimization potential")

In [ ]:
# Q3: Which products have the most negative sentiment?
print("\n❓ Q3: Products with Negative Sentiment")
print("=" * 70)

product_sentiment = reviews_df.groupby('product_name').agg({
    'rating': ['mean', 'count'],
    'sentiment': lambda x: (x == 'negative').sum() / len(x) * 100
}).round(2)

product_sentiment.columns = ['avg_rating', 'review_count', 'negative_pct']
product_sentiment = product_sentiment.sort_values('negative_pct', ascending=False)

print(product_sentiment)
print(f"\n💡 Insight: {product_sentiment.index[0]} has highest negative sentiment ({product_sentiment.iloc[0]['negative_pct']:.1f}%)")
print(f"   This product should be priority for discount optimization!")

In [ ]:
# Q4: Look at inventory and sales data
print("\n❓ Q4: Inventory & Sales Overview")
print("=" * 70)

print("\n📦 Inventory Sample:")
print(inventory_df.head())
print(f"\nDate range: {inventory_df['date'].min()} to {inventory_df['date'].max()}")

print("\n💰 Sales Sample:")
print(sales_df.head())
print(f"\nDate range: {sales_df['date'].min()} to {sales_df['date'].max()}")

In [ ]:
# Calculate average stock and sales by product
avg_stock = inventory_df.groupby('product_name')['stock_level'].mean().sort_values(ascending=False)
avg_sales = sales_df.groupby('product_name')['sales_volume'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

avg_stock.plot(kind='barh', ax=axes[0], color='lightgreen')
axes[0].set_title('Average Stock Level by Product', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Average Stock')

avg_sales.plot(kind='barh', ax=axes[1], color='lightblue')
axes[1].set_title('Average Daily Sales by Product', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Average Sales Volume')

plt.tight_layout()
plt.show()

## Step 3: Data Quality Check

In [ ]:
print("\n🔍 DATA QUALITY ASSESSMENT")
print("=" * 70)

# Check missing values
print("\n📊 Missing Values:")
print("\nReviews:")
print(reviews_df.isnull().sum())
print("\nInventory:")
print(inventory_df.isnull().sum())
print("\nSales:")
print(sales_df.isnull().sum())

# Check duplicates
print(f"\n🔄 Duplicate Reviews: {reviews_df.duplicated().sum()}")
print(f"🔄 Duplicate Inventory Records: {inventory_df.duplicated().sum()}")
print(f"🔄 Duplicate Sales Records: {sales_df.duplicated().sum()}")

if reviews_df.isnull().sum().sum() > 0 or reviews_df.duplicated().sum() > 0:
    print("\n⚠️  Data quality issues detected - need cleaning in ETL phase!")
else:
    print("\n✅ Data quality looks good!")

## Step 4: Feasibility Conclusion

Based on our analysis, let's determine if this project is feasible:

In [ ]:
print("\n" + "=" * 70)
print("🎯 PROJECT FEASIBILITY ASSESSMENT")
print("=" * 70)

feasibility_score = 0
max_score = 4

# Check 1: Sufficient data
if review_counts.min() > 50:
    print("\n✅ Sufficient review data per product")
    feasibility_score += 1
else:
    print("\n❌ Insufficient review data")

# Check 2: Sentiment variation
if sentiment_dist.get('negative', 0) > 10:
    print("✅ Enough negative sentiment to optimize")
    feasibility_score += 1
else:
    print("❌ Too little negative sentiment")

# Check 3: Inventory data available
if len(inventory_df) > 0:
    print("✅ Inventory data available")
    feasibility_score += 1

# Check 4: Sales data available
if len(sales_df) > 0:
    print("✅ Sales data available")
    feasibility_score += 1

print(f"\n📊 Feasibility Score: {feasibility_score}/{max_score}")

if feasibility_score >= 3:
    print("\n🎉 RECOMMENDATION: GO - Project is feasible!")
    print("\nNext Steps:")
    print("1. Clean and prepare data (ETL)")
    print("2. Build sentiment analysis model")
    print("3. Create discount recommendation system")
    print("4. Calculate business impact")
else:
    print("\n⚠️  RECOMMENDATION: NO-GO - Need more/better data")

print("\n" + "=" * 70)

## 💡 Key Takeaways

1. **Always start with business analysis** - Don't build ML models without understanding the problem
2. **Data quality matters** - Identify issues early before investing in complex solutions
3. **Check feasibility** - Not every problem needs ML; sometimes simple rules work better
4. **Understand the data** - Know what you have before deciding what to build

## 🚀 Next: ETL & Data Preparation
